In [54]:
import pandas as pd
import numpy as np
import re

In [55]:
# 1. Load the primary dataset
file_path = r'C:\Users\DELL\Desktop\pak_startup\Funded startups database (Pakistan).xlsx'
df = pd.read_excel(file_path)

In [56]:
# 2. Function to clean currency and convert to float
def clean_currency(value):
    if pd.isna(value) or str(value).strip() in ['—', '-', 'Undisclosed', 'undisclosed amount', '']:
        return np.nan
    
    # Remove symbols and whitespace
    value = str(value).replace('$', '').replace('PKR', '').replace('€', '').replace(',', '').strip()
    
    # Handle 'M' for Millions and 'K' for Thousands
    multiplier = 1
    if 'M' in value:
        multiplier = 1_000_000
        value = value.replace('M', '')
    elif 'K' in value:
        multiplier = 1_000
        value = value.replace('K', '')
        
    try:
        return float(value) * multiplier
    except ValueError:
        return np.nan

In [57]:
# 3. Standardize Date Formats
def clean_date(date_str):
    if pd.isna(date_str) or str(date_str).strip() in ['—', '-', '']:
        return pd.NaT
    # Attempt to parse various formats (2020-03-15, 15-03-20, 06/31/2018)
    return pd.to_datetime(date_str, errors='coerce')


In [58]:
# 4. Apply cleaning to the DataFrame

# Clean Headers (Lowercase and replace spaces with underscores)
df.columns = [col.lower().replace(' ', '_').replace('(', '').replace(')', '') for col in df.columns]


# Clean Funding Columns
funding_cols = [col for col in df.columns if 'funding_round' in col and 'investment' in col]
for col in funding_cols:
    df[col] = df[col].apply(clean_currency)
    

# Clean Date Columns
date_cols = [col for col in df.columns if 'date' in col]
for col in date_cols:
    df[col] = df[col].apply(clean_date)



In [59]:
# 5. Handle missing values in Founders (Fill '-' with NaN)
df = df.replace('-', np.nan)

C:\Users\DELL\AppData\Local\Temp\ipykernel_15180\2893257112.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('-', np.nan)


In [60]:
# 6. Drop redundant empty columns (like the one found after Founder 5)
df = df.dropna(axis=1, how='all')

In [61]:
# 7. Dropping unnecessary cols
cols_to_drop = ['last_edited_by', 'website', 'social_media_urls', 'email', 'phone']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [62]:
cols_to_drop = [
'funding_round_3_investment','founder_round_3_investors','funding_round_3_date',
'funding_round_4_investment','founder_round_4_investors','funding_round_4_date'
]

df.drop(columns=cols_to_drop, inplace=True)

In [63]:
# 8. Standardize text in Categorical columns
categorical_cols = ['industry_vertical', 'funding_status', 'operating_status']
for col in [c for c in categorical_cols if c in df.columns]:
    df[col] = df[col].str.strip().str.title()


In [64]:
# 9. Extract Primary Industry (take the first category before the comma)
if 'industry_vertical' in df.columns:
    df['primary_industry'] = df['industry_vertical'].str.split(',').str[0]

In [65]:
# 10. Handle Duplicate Startups (keep the one with the most data)
# Sort by number of non-null values and keep the best record
df['null_count'] = df.isnull().sum(axis=1)
df = df.sort_values('null_count').drop_duplicates(subset=['startup_name'], keep='first')
df = df.drop(columns=['null_count'])

In [66]:
# 11. Fill remaining NaN in specific columns with "Unknown" 
# (Better for Power BI filters than 'NaN')
cols_to_fill = ['primary_industry', 'location', 'hq']
for col in [c for c in cols_to_fill if c in df.columns]:
    df[col] = df[col].fillna('Unknown')

In [67]:


# 12. Fix the 1970 Date Bug
def fix_epoch_dates(val):
    if pd.isna(val): return val
    val_str = str(val)
    if '1970-01-01' in val_str and '.' in val_str:
        # Extract the year from the microsecond part (e.g., ...0002018 -> 2018)
        year = val_str.split('.')[-1][-4:]
        return f"{year}-01-01"
    return val_str

df['founded_date'] = df['founded_date'].apply(fix_epoch_dates)
df['founded_date'] = pd.to_datetime(df['founded_date'], errors='coerce')



In [68]:
# 13. Unify HQ / Region Names
hq_mapping = {
    'APAC': 'Asia-Pacific',
    'Asia-Pacific (APAC)': 'Asia-Pacific'
}
df['hq'] = df['hq'].replace(hq_mapping)


In [69]:
# 14. Standardize Cities (Location)
def clean_city(loc):
    if pd.isna(loc) or loc == 'Unknown': return 'Unknown'
    # Take the first word, remove commas and extra spaces
    city = str(loc).split(',')[0].strip()
    # Manual fix for neighborhood names (e.g., Gulberg is in Lahore)
    if city == 'Gulberg': return 'Lahore'
    return city

df['city'] = df['location'].apply(clean_city)


In [70]:
# 15. Drop nearly-empty columns (Founders 4 and 5)
df = df.drop(columns=['founder_4', 'founder_5'], errors='ignore')


In [71]:
# 16. Final String Strip (Remove hidden spaces that ruin grouping)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)


In [72]:
# 1. Define the specific "bad" strings
noise_words = ['Pakistan', 'Kili', 'Unknown', 'Unknown City']

# 2. Filter out rows where the city contains ANY of those words
# We use case=False to catch 'pakistan', 'PAKISTAN', etc.
df = df[~df['city'].str.contains('|'.join(noise_words), na=False, case=False)]

# 3. Final cleanup: strip any extra spaces and capitalize
df['city'] = df['city'].str.strip().str.title()

# 4. Verify the output
print(f"Remaining unique cities: {df['city'].nunique()}")
print(df['city'].unique())

Remaining unique cities: 8
['Karachi' 'Lahore' 'Islamabad' 'Hyderabad' 'Sialkot' 'Gujrat'
 'Faisalabad' 'Rawalpindi']


In [73]:
df['founders'] = df[['founder_1','founder_2','founder_3']]\
.fillna('').agg(', '.join, axis=1)

In [74]:
df['founders'] = df['founders'].str.strip(', ')


In [75]:
df.drop(columns=['founder_1','founder_2','founder_3'], inplace=True)

In [76]:
df.columns

Index(['startup_name', 'active_opinion_by_cms', 'location', 'hq',
       'founded_date', 'industry_vertical', 'horizontal', 'operating_status',
       'funding_status', 'employees', 'funding_round_1_investment',
       'founder_round_1_investors', 'funding_round_1_date',
       'funding_round_2_investment', 'founder_round_2_investors',
       'funding_round_2_date', 'primary_industry', 'city', 'founders'],
      dtype='object')

In [77]:
df['city'] = df['city'].str.title()

In [79]:
# Save the absolute final version
df.to_csv('final_pak_startups_for_analysis.csv', index=False)
print("Data is now 100% clean!")

Data is now 100% clean!
